Проверяем данные data/final/ml_data_final

In [2]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

DATA_PATH = Path("C:/Users/Ева/Desktop/Хакатон Альфа/AURELIUS/scoring/data/final/ml_data_final").resolve()

parquet_pattern = (
    DATA_PATH.as_posix().rstrip("/") + "/**/*.parquet"
)

con = duckdb.connect()
con.execute("SET threads TO 4")

query = """
WITH episodes AS (
    SELECT
        borrower_id,
        episode_id,

        -- Признаки на момент выдачи
        FIRST(origination_month_index ORDER BY month_on_book)
            AS origination_month_index,
        FIRST(age_months ORDER BY month_on_book) AS age_months,
        FIRST(industry ORDER BY month_on_book) AS industry,
        FIRST(entity_type ORDER BY month_on_book) AS entity_type,
        FIRST(monthly_revenue ORDER BY month_on_book) AS monthly_revenue,
        FIRST(revenue_trend_12m ORDER BY month_on_book) AS revenue_trend_12m,
        FIRST(revenue_volatility ORDER BY month_on_book) AS revenue_volatility,
        FIRST(operating_margin ORDER BY month_on_book) AS operating_margin,
        FIRST(cash_buffer_months ORDER BY month_on_book) AS cash_buffer_months,
        FIRST(dscr ORDER BY month_on_book) AS dscr,
        FIRST(debt_to_revenue ORDER BY month_on_book) AS debt_to_revenue,
        FIRST(prior_overdue_30d ORDER BY month_on_book) AS prior_overdue_30d,
        FIRST(requested_amount ORDER BY month_on_book) AS requested_amount,
        FIRST(term_months ORDER BY month_on_book) AS term_months,
        FIRST(annual_rate ORDER BY month_on_book) AS annual_rate,
        FIRST(macro_stress ORDER BY month_on_book) AS macro_stress,

        -- История наблюдений
        MIN(month_on_book) AS first_month,
        MAX(month_on_book) AS last_month,
        COUNT(*) AS observation_count,
        COUNT(DISTINCT month_on_book) AS distinct_months,

        MIN(month_on_book)
            FILTER (WHERE default_event = 1)
            AS first_default_month

    FROM read_parquet(?, union_by_name = true)
    GROUP BY borrower_id, episode_id
)

SELECT
    *,
    COALESCE(first_default_month, last_month) AS duration,
    CASE
        WHEN first_default_month IS NOT NULL THEN 1
        ELSE 0
    END AS event
FROM episodes
"""

df = con.execute(query, [parquet_pattern]).df()

print("Размер:", df.shape)
print("Уникальных заёмщиков:", df.borrower_id.nunique())
print("Дефолтов:", int(df.event.sum()))

display(df.head())

Размер: (400000, 25)
Уникальных заёмщиков: 200000
Дефолтов: 57279


,borrower_id,episode_id,origination_month_index,age_months,industry,entity_type,monthly_revenue,revenue_trend_12m,revenue_volatility,operating_margin,...,term_months,annual_rate,macro_stress,first_month,last_month,observation_count,distinct_months,first_default_month,duration,event
0,4280,4280,81,135,manufacturing,IP,4.957230e+05,-0.292607,0.149326,0.198391,...,24,0.208708,1.108760,1,24,49,24,<NA>,24,0
1,4306,4306,75,94,retail,OOO,5.137438e+06,-0.016607,0.119752,0.150612,...,36,0.164039,0.978443,1,4,9,4,4,4,1
2,8088,8088,79,70,manufacturing,OOO,7.752908e+06,-0.019538,0.199331,0.135560,...,36,0.197065,1.073867,1,36,73,36,<NA>,36,0
3,8540,8540,81,86,agriculture,IP,2.819971e+06,-0.238483,0.217308,0.132649,...,48,0.212848,1.108760,1,31,63,31,<NA>,31,0
4,13107,13107,75,85,construction,OOO,3.443500e+06,-0.043291,0.225677,0.023565,...,12,0.185718,0.978443,1,12,25,12,<NA>,12,0


In [3]:
checks = {
    "Эпизоды начинаются не с месяца 1":
        (df.first_month != 1).sum(),

    "Пропущены месяцы внутри эпизода":
        (
            df.distinct_months != df.last_month
        ).sum(),

    "Наблюдение длиннее срока кредита":
        (
            df.last_month > df.term_months
        ).sum(),

    "Дефолт после окончания срока кредита":
        (
            (df.event == 1) &
            (df.duration > df.term_months)
        ).sum(),

    "Некорректная длительность":
        (df.duration <= 0).sum(),

    "Неполные наблюдения без дефолта":
        (
            (df.event == 0) &
            (df.last_month < df.term_months)
        ).sum()
}

display(
    pd.DataFrame(
        checks.items(),
        columns=["Проверка", "Количество"]
    )
)

,Проверка,Количество
0,Эпизоды начинаются не с месяца 1,0
1,Пропущены месяцы внутри эпизода,0
2,Наблюдение длиннее срока кредита,0
3,Дефолт после окончания срока кредита,0
4,Некорректная длительность,0
5,Неполные наблюдения без дефолта,44292


In [4]:
print("Распределение сроков кредита:")

display(
    df.groupby("term_months")
      .agg(
          episodes=("episode_id", "count"),
          defaults=("event", "sum"),
          mean_duration=("duration", "mean")
      )
      .assign(
          default_rate=lambda x:
              x["defaults"] / x["episodes"]
      )
      .reset_index()
)

print("\nСтатистика длительности:")
display(df["duration"].describe())

print("\nПропуски:")
display(
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .head(25)
)

Распределение сроков кредита:


,term_months,episodes,defaults,mean_duration,default_rate
0,12,32087,4469,11.008633,0.139278
1,18,25990,5067,15.468642,0.19496
2,24,48017,12130,19.042527,0.252619
3,36,48075,16322,24.52156,0.339511
4,48,28048,11198,28.025635,0.399244
5,60,17783,8093,29.770286,0.455098



Статистика длительности:


count     200000.0
mean     20.819855
std      12.372273
min            1.0
25%           12.0
50%           18.0
75%           25.0
max           60.0
Name: duration, dtype: Float64


Пропуски:


first_default_month        342721
entity_type                200000
origination_month_index    200000
age_months                 200000
industry                   200000
revenue_trend_12m          200000
monthly_revenue            200000
revenue_volatility         200000
episode_id                 200000
annual_rate                200000
operating_margin           200000
dscr                       200000
cash_buffer_months         200000
prior_overdue_30d          200000
requested_amount           200000
term_months                200000
debt_to_revenue            200000
duration                   200000
first_month                200000
last_month                 200000
borrower_id                     0
observation_count               0
macro_stress                    0
distinct_months                 0
event                           0
dtype: int64

In [5]:
{
    "term_months": 24,
    "months": list(range(1, 25)),
    "cumulative_pd": [
        # 24 прогнозных значения
    ]
}

{'term_months': 24,
 'months': [1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19,
  20,
  21,
  22,
  23,
  24],
 'cumulative_pd': []}

In [6]:
from pathlib import Path

# Используем путь из первого рабочего скрипта
print("DATA_PATH:", DATA_PATH)

parquet_files = sorted(DATA_PATH.rglob("*.parquet"))

print("Найдено Parquet-файлов:", len(parquet_files))

assert len(parquet_files) > 0, "Parquet-файлы не найдены"

DATA_PATH: C:\Users\Ева\Desktop\Хакатон Альфа\AURELIUS\scoring\data\final\ml_data_final
Найдено Parquet-файлов: 1600


In [7]:
import duckdb
import pandas as pd

# Используем уже созданные con и parquet_files

diagnostics = con.execute("""
WITH base AS (
    SELECT
        borrower_id,
        episode_id,
        month_on_book,
        term_months,
        monthly_revenue,
        industry,
        default_event
    FROM read_parquet(?, union_by_name=true)
),

episode_stats AS (
    SELECT
        borrower_id,
        episode_id,

        COUNT(*) AS rows_count,
        COUNT(DISTINCT month_on_book) AS months_count,

        COUNT(*) FILTER (
            WHERE month_on_book = 1
        ) AS first_month_rows,

        COUNT(*) FILTER (
            WHERE month_on_book = 1
              AND monthly_revenue IS NULL
        ) AS first_month_null_revenue,

        COUNT(*) FILTER (
            WHERE monthly_revenue IS NOT NULL
        ) AS valid_revenue_rows,

        COUNT(DISTINCT term_months) AS unique_terms,

        COUNT(*) FILTER (
            WHERE default_event = 1
        ) AS default_rows

    FROM base
    GROUP BY borrower_id, episode_id
)

SELECT
    COUNT(*) AS total_episodes,

    COUNT(*) FILTER (
        WHERE rows_count > months_count
    ) AS episodes_with_duplicate_months,

    COUNT(*) FILTER (
        WHERE first_month_rows > 1
    ) AS episodes_with_multiple_first_month_rows,

    COUNT(*) FILTER (
        WHERE first_month_null_revenue > 0
    ) AS episodes_with_null_first_month_revenue,

    COUNT(*) FILTER (
        WHERE valid_revenue_rows = 0
    ) AS episodes_without_revenue,

    COUNT(*) FILTER (
        WHERE unique_terms > 1
    ) AS episodes_with_changing_term,

    COUNT(*) FILTER (
        WHERE default_rows > 1
    ) AS episodes_with_multiple_default_rows

FROM episode_stats
""", [[str(p) for p in parquet_files]]).df()

display(diagnostics)

,total_episodes,episodes_with_duplicate_months,episodes_with_multiple_first_month_rows,episodes_with_null_first_month_revenue,episodes_without_revenue,episodes_with_changing_term,episodes_with_multiple_default_rows
0,400000,400000,200000,0,200000,0,57279


In [8]:
import pandas as pd

files = [str(p) for p in parquet_files]

# 1. Проверка идентификаторов
id_stats = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (
            WHERE borrower_id IS NULL
        ) AS null_borrower_id,
        COUNT(*) FILTER (
            WHERE episode_id IS NULL
        ) AS null_episode_id,
        COUNT(DISTINCT borrower_id) AS unique_borrowers,
        COUNT(DISTINCT (borrower_id, episode_id))
            AS unique_non_null_pairs
    FROM read_parquet(?, union_by_name=true)
""", [files]).df()

print("Проверка идентификаторов:")
display(id_stats)


# 2. Примеры строк для заёмщиков с NULL episode_id
sample_borrowers = (
    df.loc[df["episode_id"].isna(), "borrower_id"]
      .dropna()
      .head(3)
      .astype(int)
      .tolist()
)

for borrower in sample_borrowers:

    print(f"\nЗаёмщик: {borrower}")

    sample = con.execute("""
        SELECT
            borrower_id,
            episode_id,
            month_on_book,
            monthly_revenue,
            current_monthly_revenue,
            term_months,
            default_event
        FROM read_parquet(?, union_by_name=true)
        WHERE borrower_id = ?
        ORDER BY episode_id NULLS LAST, month_on_book
        LIMIT 30
    """, [files, borrower]).df()

    display(sample)

Проверка идентификаторов:


,total_rows,null_borrower_id,null_episode_id,unique_borrowers,unique_non_null_pairs
0,13327942,0,4800000,200000,400000



Заёмщик: 3063


,borrower_id,episode_id,month_on_book,monthly_revenue,current_monthly_revenue,term_months,default_event
0,3063,3063,1,2.732909e+06,NaN,24,0
1,3063,3063,1,2.732909e+06,2.732909e+06,24,0
2,3063,3063,2,2.732909e+06,2.396925e+06,24,0
3,3063,3063,2,2.732909e+06,NaN,24,0
4,3063,3063,3,2.732909e+06,NaN,24,0
5,3063,3063,3,2.732909e+06,2.454039e+06,24,0
6,3063,3063,4,2.732909e+06,NaN,24,0
7,3063,3063,4,2.732909e+06,2.336640e+06,24,0
8,3063,3063,5,2.732909e+06,NaN,24,0
9,3063,3063,5,2.732909e+06,2.396924e+06,24,0



Заёмщик: 3078


,borrower_id,episode_id,month_on_book,monthly_revenue,current_monthly_revenue,term_months,default_event
0,3078,3078,1,2.171811e+06,NaN,48,0
1,3078,3078,1,2.171811e+06,2.171811e+06,48,0
2,3078,3078,2,2.171811e+06,2.047746e+06,48,0
3,3078,3078,2,2.171811e+06,NaN,48,0
4,3078,3078,3,2.171811e+06,NaN,48,0
5,3078,3078,3,2.171811e+06,2.008569e+06,48,0
6,3078,3078,4,2.171811e+06,NaN,48,0
7,3078,3078,4,2.171811e+06,2.095566e+06,48,0
8,3078,3078,5,2.171811e+06,NaN,48,0
9,3078,3078,5,2.171811e+06,1.845766e+06,48,0



Заёмщик: 3179


,borrower_id,episode_id,month_on_book,monthly_revenue,current_monthly_revenue,term_months,default_event
0,3179,3179,1,1.055598e+06,1.055598e+06,12,0
1,3179,3179,1,1.055598e+06,NaN,12,0
2,3179,3179,2,1.055598e+06,1.104491e+06,12,0
3,3179,3179,2,1.055598e+06,NaN,12,0
4,3179,3179,3,1.055598e+06,1.356585e+06,12,0
5,3179,3179,3,1.055598e+06,NaN,12,0
6,3179,3179,4,1.055598e+06,1.371742e+06,12,0
7,3179,3179,4,1.055598e+06,NaN,12,0
8,3179,3179,5,1.055598e+06,1.373111e+06,12,0
9,3179,3179,5,1.055598e+06,NaN,12,0


In [9]:
files = [str(p) for p in parquet_files]

result = con.execute("""
SELECT
    CASE
        WHEN episode_id IS NULL THEN 'NULL episode_id'
        ELSE 'VALID episode_id'
    END AS record_type,

    COUNT(*) AS rows_count,

    COUNT(DISTINCT borrower_id) AS borrowers,

    COUNT(DISTINCT episode_id) AS episodes,

    COUNT(*) FILTER (
        WHERE month_on_book IS NULL
    ) AS null_month,

    COUNT(*) FILTER (
        WHERE monthly_revenue IS NULL
    ) AS null_revenue,

    COUNT(*) FILTER (
        WHERE current_monthly_revenue IS NULL
    ) AS null_current_revenue,

    COUNT(*) FILTER (
        WHERE default_event IS NULL
    ) AS null_default

FROM read_parquet(?, union_by_name=true)

GROUP BY 1
ORDER BY 1
""", [files]).df()

display(result)

,record_type,rows_count,borrowers,episodes,null_month,null_revenue,null_current_revenue,null_default
0,NULL episode_id,4800000,200000,0,4800000,4800000,4800000,4800000
1,VALID episode_id,8527942,200000,200000,200000,200000,4363971,200000


In [10]:
result = con.execute("""
WITH monthly AS (
    SELECT
        borrower_id,
        episode_id,
        month_on_book,
        COUNT(*) AS row_count
    FROM read_parquet(?, union_by_name=true)
    WHERE episode_id IS NOT NULL
      AND month_on_book IS NOT NULL
    GROUP BY 1, 2, 3
)
SELECT
    row_count,
    COUNT(*) AS month_groups
FROM monthly
GROUP BY row_count
ORDER BY row_count
""", [files]).df()

display(result)

,row_count,month_groups
0,2,4163971


In [11]:
files = [str(p) for p in parquet_files]

result = con.execute("""
WITH valid AS (
    SELECT *
    FROM read_parquet(?, union_by_name=true)
    WHERE episode_id IS NOT NULL
      AND month_on_book IS NOT NULL
),

monthly AS (
    SELECT
        borrower_id,
        episode_id,
        month_on_book,

        COUNT(*) AS rows_count,

        COUNT(DISTINCT monthly_revenue)
            AS revenue_variants,

        COUNT(DISTINCT dscr)
            AS dscr_variants,

        COUNT(DISTINCT term_months)
            AS term_variants,

        COUNT(DISTINCT default_event)
            AS default_variants,

        COUNT(*) FILTER (
            WHERE current_monthly_revenue IS NOT NULL
        ) AS rows_with_current_revenue

    FROM valid
    GROUP BY 1, 2, 3
)

SELECT
    COUNT(*) AS unique_months,

    COUNT(*) FILTER (
        WHERE rows_count = 2
    ) AS months_with_two_rows,

    COUNT(*) FILTER (
        WHERE revenue_variants > 1
    ) AS conflicting_revenue,

    COUNT(*) FILTER (
        WHERE dscr_variants > 1
    ) AS conflicting_dscr,

    COUNT(*) FILTER (
        WHERE term_variants > 1
    ) AS conflicting_terms,

    COUNT(*) FILTER (
        WHERE default_variants > 1
    ) AS conflicting_defaults,

    COUNT(*) FILTER (
        WHERE rows_with_current_revenue = 1
    ) AS months_with_one_current_record

FROM monthly
""", [files]).df()

display(result)

,unique_months,months_with_two_rows,conflicting_revenue,conflicting_dscr,conflicting_terms,conflicting_defaults,months_with_one_current_record
0,4163971,4163971,0,0,0,0,4163971


In [12]:
from pathlib import Path
import duckdb
import pandas as pd
import numpy as np

# ============================================================
# 1. ПУТИ
# ============================================================

SCORING_DIR = Path(
    r"C:\Users\Ева\Desktop\Хакатон Альфа\AURELIUS\scoring"
)

SOURCE_DIR = SCORING_DIR / "data" / "final" / "ml_data_final"
OUTPUT_DIR = SCORING_DIR / "data" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "ml_survival_dataset.parquet"

parquet_files = sorted(SOURCE_DIR.rglob("*.parquet"))

print("Исходный датасет:", SOURCE_DIR)
print("Найдено файлов:", len(parquet_files))
print("Итоговый файл:", OUTPUT_FILE)

assert len(parquet_files) == 1600, (
    f"Ожидалось 1600 Parquet-файлов, найдено {len(parquet_files)}"
)

# ============================================================
# 2. DUCKDB
# ============================================================

con = duckdb.connect()
con.execute("SET threads TO 4")

files = [str(p) for p in parquet_files]

# ============================================================
# 3. ПОДГОТОВКА КРЕДИТНЫХ ЭПИЗОДОВ
# ============================================================

query = """
WITH valid_monthly AS (

    SELECT
        borrower_id,
        episode_id,
        origination_month_index,
        month_on_book,
        age_months,
        industry,
        entity_type,
        monthly_revenue,
        revenue_trend_12m,
        revenue_volatility,
        operating_margin,
        cash_buffer_months,
        dscr,
        debt_to_revenue,
        prior_overdue_30d,
        requested_amount,
        term_months,
        annual_rate,
        macro_stress,
        default_event

    FROM read_parquet(?, union_by_name=true)

    WHERE borrower_id IS NOT NULL
      AND episode_id IS NOT NULL
      AND month_on_book IS NOT NULL
      AND current_monthly_revenue IS NOT NULL
),

episodes AS (

    SELECT
        borrower_id,
        episode_id,

        -- Исходные признаки на момент заявки
        FIRST(origination_month_index ORDER BY month_on_book)
            AS origination_month_index,

        FIRST(age_months ORDER BY month_on_book)
            AS age_months,

        FIRST(industry ORDER BY month_on_book)
            AS industry,

        FIRST(entity_type ORDER BY month_on_book)
            AS entity_type,

        FIRST(monthly_revenue ORDER BY month_on_book)
            AS monthly_revenue,

        FIRST(revenue_trend_12m ORDER BY month_on_book)
            AS revenue_trend_12m,

        FIRST(revenue_volatility ORDER BY month_on_book)
            AS revenue_volatility,

        FIRST(operating_margin ORDER BY month_on_book)
            AS operating_margin,

        FIRST(cash_buffer_months ORDER BY month_on_book)
            AS cash_buffer_months,

        FIRST(dscr ORDER BY month_on_book)
            AS dscr,

        FIRST(debt_to_revenue ORDER BY month_on_book)
            AS debt_to_revenue,

        FIRST(prior_overdue_30d ORDER BY month_on_book)
            AS prior_overdue_30d,

        FIRST(requested_amount ORDER BY month_on_book)
            AS requested_amount,

        FIRST(term_months ORDER BY month_on_book)
            AS term_months,

        FIRST(annual_rate ORDER BY month_on_book)
            AS annual_rate,

        FIRST(macro_stress ORDER BY month_on_book)
            AS macro_stress,

        -- Временные характеристики
        MIN(month_on_book) AS first_month,
        MAX(month_on_book) AS last_month,

        COUNT(*) AS observation_count,

        MIN(month_on_book)
            FILTER (WHERE default_event = 1)
            AS first_default_month

    FROM valid_monthly

    GROUP BY borrower_id, episode_id
)

SELECT
    borrower_id,
    episode_id,

    origination_month_index,
    age_months,
    industry,
    entity_type,

    monthly_revenue,
    revenue_trend_12m,
    revenue_volatility,
    operating_margin,
    cash_buffer_months,
    dscr,
    debt_to_revenue,
    prior_overdue_30d,

    requested_amount,
    term_months,
    annual_rate,
    macro_stress,

    first_month,
    last_month,
    observation_count,
    first_default_month,

    COALESCE(first_default_month, last_month)
        AS duration,

    CASE
        WHEN first_default_month IS NOT NULL THEN 1
        ELSE 0
    END AS event

FROM episodes
"""

df_survival = con.execute(query, [files]).df()

# ============================================================
# 4. ПРОВЕРКА
# ============================================================

print("\n" + "=" * 60)
print("ИТОГОВЫЙ ОБУЧАЮЩИЙ ДАТАСЕТ")
print("=" * 60)

print("Размер:", df_survival.shape)
print("Заёмщиков:", df_survival["borrower_id"].nunique())
print("Кредитных эпизодов:", len(df_survival))
print("Дефолтов:", int(df_survival["event"].sum()))
print(
    "Без дефолта:",
    int((df_survival["event"] == 0).sum())
)

print("\nПропуски:")
print(df_survival.isna().sum().sort_values(ascending=False))

# ============================================================
# 5. КОНТРОЛЬ КАЧЕСТВА
# ============================================================

assert not df_survival.empty, "Датасет пуст!"

assert not df_survival.duplicated(
    ["borrower_id", "episode_id"]
).any(), "Обнаружены повторяющиеся кредитные эпизоды!"

assert (df_survival["duration"] > 0).all(), (
    "Есть некорректная длительность!"
)

assert (
    df_survival["duration"] <= df_survival["term_months"]
).all(), "Длительность превышает срок кредита!"

assert (
    df_survival["first_month"] == 1
).all(), "Есть эпизоды, начинающиеся не с первого месяца!"

# Проверка обязательных признаков
feature_columns = [
    "age_months",
    "industry",
    "entity_type",
    "monthly_revenue",
    "revenue_trend_12m",
    "revenue_volatility",
    "operating_margin",
    "cash_buffer_months",
    "dscr",
    "debt_to_revenue",
    "prior_overdue_30d",
    "requested_amount",
    "term_months",
    "annual_rate",
    "macro_stress",
]

missing_features = df_survival[feature_columns].isna().sum()

assert missing_features.sum() == 0, (
    f"Пропуски в признаках:\n{missing_features[missing_features > 0]}"
)

# ============================================================
# 6. СОХРАНЕНИЕ
# ============================================================

df_survival.to_parquet(
    OUTPUT_FILE,
    engine="pyarrow",
    compression="zstd",
    index=False
)

print("\nДатасет успешно сохранён!")
print("Путь:", OUTPUT_FILE)
print(
    "Размер файла:",
    round(OUTPUT_FILE.stat().st_size / 1024**2, 2),
    "МБ"
)

display(df_survival.head())

Исходный датасет: C:\Users\Ева\Desktop\Хакатон Альфа\AURELIUS\scoring\data\final\ml_data_final
Найдено файлов: 1600
Итоговый файл: C:\Users\Ева\Desktop\Хакатон Альфа\AURELIUS\scoring\data\processed\ml_survival_dataset.parquet

ИТОГОВЫЙ ОБУЧАЮЩИЙ ДАТАСЕТ
Размер: (200000, 24)
Заёмщиков: 200000
Кредитных эпизодов: 200000
Дефолтов: 57279
Без дефолта: 142721

Пропуски:
first_default_month        142721
borrower_id                     0
origination_month_index         0
episode_id                      0
industry                        0
entity_type                     0
monthly_revenue                 0
age_months                      0
revenue_trend_12m               0
revenue_volatility              0
cash_buffer_months              0
operating_margin                0
debt_to_revenue                 0
prior_overdue_30d               0
requested_amount                0
dscr                            0
term_months                     0
annual_rate                     0
first_month          

,borrower_id,episode_id,origination_month_index,age_months,industry,entity_type,monthly_revenue,revenue_trend_12m,revenue_volatility,operating_margin,...,requested_amount,term_months,annual_rate,macro_stress,first_month,last_month,observation_count,first_default_month,duration,event
0,1125,1125,82,106,services,IP,1.773904e+06,-0.517334,0.265675,0.090343,...,5.070370e+06,24,0.194611,1.122448,1,8,8,<NA>,8,0
1,1140,1140,75,17,retail,OOO,5.867863e+06,0.153859,0.249888,0.142453,...,1.357590e+07,48,0.139471,0.978443,1,31,31,<NA>,31,0
2,1576,1576,81,114,services,IP,1.903982e+06,-0.078668,0.150103,-0.006643,...,2.666259e+06,36,0.177015,1.108760,1,36,36,<NA>,36,0
3,1745,1745,72,56,construction,OOO,2.657344e+06,-0.244306,0.171109,0.054196,...,1.711685e+06,24,0.209074,0.090746,1,5,5,5,5,1
4,1923,1923,72,68,retail,OOO,3.211327e+06,-0.199073,0.133975,0.086946,...,4.839822e+06,36,0.174739,0.090746,1,10,10,10,10,1


Проверяем датасет

In [13]:
df_check = pd.read_parquet(OUTPUT_FILE)

print("Файл:", OUTPUT_FILE)
print("Размер:", df_check.shape)

print("\nТипы данных:")
print(df_check.dtypes)

print("\nРаспределение событий:")
print(df_check["event"].value_counts())

print("\nРаспределение сроков:")
display(
    df_check.groupby("term_months")
    .agg(
        episodes=("episode_id", "count"),
        defaults=("event", "sum"),
        mean_duration=("duration", "mean")
    )
    .reset_index()
)

print("\nПервые строки:")
display(df_check.head())

Файл: C:\Users\Ева\Desktop\Хакатон Альфа\AURELIUS\scoring\data\processed\ml_survival_dataset.parquet
Размер: (200000, 24)

Типы данных:
borrower_id                  int64
episode_id                   int64
origination_month_index      int64
age_months                   int64
industry                       str
entity_type                    str
monthly_revenue            float64
revenue_trend_12m          float64
revenue_volatility         float64
operating_margin           float64
cash_buffer_months         float64
dscr                       float64
debt_to_revenue            float64
prior_overdue_30d            int64
requested_amount           float64
term_months                  int64
annual_rate                float64
macro_stress               float64
first_month                  int64
last_month                   int64
observation_count            int64
first_default_month          Int64
duration                     int64
event                        int32
dtype: object

Распредел

,term_months,episodes,defaults,mean_duration
0,12,32087,4469,11.008633
1,18,25990,5067,15.468642
2,24,48017,12130,19.042527
3,36,48075,16322,24.521560
4,48,28048,11198,28.025635
5,60,17783,8093,29.770286



Первые строки:


,borrower_id,episode_id,origination_month_index,age_months,industry,entity_type,monthly_revenue,revenue_trend_12m,revenue_volatility,operating_margin,...,requested_amount,term_months,annual_rate,macro_stress,first_month,last_month,observation_count,first_default_month,duration,event
0,1125,1125,82,106,services,IP,1.773904e+06,-0.517334,0.265675,0.090343,...,5.070370e+06,24,0.194611,1.122448,1,8,8,<NA>,8,0
1,1140,1140,75,17,retail,OOO,5.867863e+06,0.153859,0.249888,0.142453,...,1.357590e+07,48,0.139471,0.978443,1,31,31,<NA>,31,0
2,1576,1576,81,114,services,IP,1.903982e+06,-0.078668,0.150103,-0.006643,...,2.666259e+06,36,0.177015,1.108760,1,36,36,<NA>,36,0
3,1745,1745,72,56,construction,OOO,2.657344e+06,-0.244306,0.171109,0.054196,...,1.711685e+06,24,0.209074,0.090746,1,5,5,5,5,1
4,1923,1923,72,68,retail,OOO,3.211327e+06,-0.199073,0.133975,0.086946,...,4.839822e+06,36,0.174739,0.090746,1,10,10,10,10,1


In [16]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/ml_survival_dataset.parquet"
)

print("Всего эпизодов:", len(df))
print("Уникальных эпизодов:", df["episode_id"].nunique())

print("\nРаспределение событий:")
print(df["event"].value_counts())

print("\nТипы завершения наблюдения:")

conditions = {
    "Дефолт": (df["event"] == 1),
    "Без дефолта, полный срок": (
        (df["event"] == 0) &
        (df["duration"] >= df["term_months"])
    ),
    "Без дефолта, наблюдение короче срока": (
        (df["event"] == 0) &
        (df["duration"] < df["term_months"])
    )
}

for name, mask in conditions.items():
    count = mask.sum()
    print(f"{name}: {count:,} ({count / len(df):.2%})")

print("\nПроверка некорректных длительностей:")

print("duration <= 0:", (df["duration"] <= 0).sum())
print(
    "duration > term_months:",
    (df["duration"] > df["term_months"]).sum()
)

print(
    "Дефолт после окончания срока:",
    (
        (df["event"] == 1) &
        (df["duration"] > df["term_months"])
    ).sum()
)

print("\nПропуски:")
print(df.isna().sum()[lambda x: x > 0])

Всего эпизодов: 200000
Уникальных эпизодов: 200000

Распределение событий:
event
0    142721
1     57279
Name: count, dtype: int64

Типы завершения наблюдения:
Дефолт: 57,279 (28.64%)
Без дефолта, полный срок: 98,429 (49.21%)
Без дефолта, наблюдение короче срока: 44,292 (22.15%)

Проверка некорректных длительностей:
duration <= 0: 0
duration > term_months: 0
Дефолт после окончания срока: 0

Пропуски:
first_default_month    142721
dtype: int64
